# Day 10 · LLM 서빙 — Colab 실습

Colab 무료 GPU(T4 · VRAM 16GB)에서 서빙 엔진을 직접 띄운다. 관리형 API 뒤의 **서빙 계층**을 코드로 확인하고, 7강 루프를 base_url 교체만으로 연결한 뒤, LangGraph 오케스트레이터까지 올린다.

> **시작 전 필수** — 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU → 저장.** 이걸 안 하면 CPU 인스턴스라 모든 랩이 느리다.

| 절 | 내용 | 결과 |
|---|---|---|
| Lab 0 | GPU 런타임 확인 | T4 16GB 확인 |
| Lab 1 | Ollama 설치 → 서빙 → API 확인 | 이 인스턴스에 OpenAI 호환 서버 |
| Lab 2 | 7강 에이전트 루프를 내 서버로 | base_url 두 줄 교체로 에이전트 동작 |
| Lab 3 (선택) | vLLM으로 같은 모델 서빙 | 엔진을 갈아끼워도 앱 코드 유지 |
| Lab 4 | LangGraph 플래너·워커·종합 | 서빙 위의 코드 오케스트레이터 |
| 실무 예제 | 고객 리뷰 일괄 분석 | 배치 병렬 → 경영 보고 |

**층 구조**

```
[내 앱 · 오케스트레이터]
     |  POST /v1/chat/completions
[서빙 엔진]   vLLM · Ollama · NIM
     |  배칭 · KV 캐시 관리
[GPU + 모델]
```

**주의** — Colab 세션이 끊기면 디스크가 초기화된다(설치·모델 다시). 무료 한도(하루 몇 시간)를 다 쓰면 쿨다운이 있다. 내 컴퓨터에 직접 설치하는 방법은 맨 아래 부록에 있다.

## Lab 0 · GPU 런타임 확인

아래 셀에서 `Tesla T4`와 `16384MiB`가 보이면 준비 완료다. `command not found`가 나오면 런타임 유형이 CPU다 — 런타임 유형 변경 후 다시 실행한다.

In [ ]:
!nvidia-smi

## Lab 1 · Ollama 서빙

Colab 인스턴스는 우분투 VM이라 리눅스 설치 스크립트를 그대로 쓴다. 설치 스크립트가 GPU(CUDA)를 자동 감지한다.

**반드시 `-instruct` 태그를 받는다.** 무늬만 같은 `qwen3:4b`(hybrid)는 thinking이 기본으로 켜져 있어, 실측에서 한 문장 답변에 2,232토큰(안 보이는 추론)을 생성했다. `-instruct` 변형은 같은 질문에 63토큰이었고, 스트리밍 첫 응답이 즉시 시작된다. `/v1`의 `think:false`·`/no_think`로는 꺼지지 않았다 — 해법은 태그 선택이다.

In [ ]:
# 설치 — Ollama 배포판이 zstd 압축이라 압축 해제 도구를 먼저 설치한다
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 서버를 백그라운드 프로세스로 띄운다
# (!ollama serve & 는 셀 종료와 함께 정리될 수 있어 Popen이 안정적이다)
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama --version

In [ ]:
!ollama pull qwen3:4b-instruct   # 약 2.5GB (Q4 양자화 GGUF)

In [ ]:
%pip install -q openai

In [ ]:
# NVIDIA API를 부르던 코드에서 base_url·model 두 줄만 다르다
from openai import OpenAI

llm = OpenAI(
    base_url="http://localhost:11434/v1",   # ← 이 인스턴스의 Ollama
    api_key="ollama",                        # 아무 문자열이면 된다
)
MODEL = "qwen3:4b-instruct"                  # 반드시 -instruct 태그

r = llm.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "서빙 엔진이 뭔지 한 문장으로."}],
)
print(r.choices[0].message.content)


In [ ]:
# 스트리밍도 같은 규격으로 동작한다
stream = llm.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "KV 캐시를 두 문장으로 설명해줘."}],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)
print()


### 관찰

- 첫 토큰까지의 시간과 생성 속도를 NVIDIA API와 비교해 본다.
- `!ollama ps` 로 모델이 메모리에 올라와 있는 것을 확인한다 (GGUF 양자화 크기). `!nvidia-smi` 를 다시 실행하면 VRAM 사용량이 늘어 있다 — 가중치가 GPU에 올라간 것이다.
- 기본 컨텍스트 길이는 짧다(2k~4k). 에이전트처럼 긴 대화를 돌릴 때는 `num_ctx` 를 늘린다 — KV 캐시 메모리가 비례해 커진다.

> 여기까지 되면 이 인스턴스가 **관리형 API와 같은 규격의 서버**다. rate limit이 없다.

## Lab 2 · base_url 교체 — 7강 루프를 내 모델로

7강에서 만든 에이전트 루프(판단 → 도구 → 관찰)를 **그대로** 가져와, 모델만 내 서버로 바꾼다.
아래는 도구 하나짜리 최소 루프다 — 바뀐 곳은 상단 두 줄뿐이다.


In [ ]:
import json

# ── 바뀐 곳: 이 두 줄이 전부 ──────────────────────
llm = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "qwen3:4b-instruct"          # Lab 1에서 받아둔 모델 재사용
# ─────────────────────────────────────────────

# 도구 하나 (7강과 같은 형식)
TASKS = []
def add_task(title: str) -> dict:
    TASKS.append({"id": len(TASKS) + 1, "title": title})
    return TASKS[-1]

TOOLS = [{
    "type": "function",
    "function": {
        "name": "add_task",
        "description": "할 일을 추가한다",
        "parameters": {
            "type": "object",
            "properties": {"title": {"type": "string"}},
            "required": ["title"],
        },
    },
}]

def run_agent(question: str, max_steps: int = 5) -> str:
    messages = [
        {"role": "system", "content": "필요하면 도구로 처리한 뒤 한국어로 간결히 답하라."},
        {"role": "user", "content": question},
    ]
    for _ in range(max_steps):
        m = llm.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, temperature=0.2,
        ).choices[0].message
        if not m.tool_calls:                       # 도구를 안 부르면 종료
            return m.content
        messages.append({"role": "assistant", "content": m.content or "",
                         "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"  [로컬 모델 → 도구] {tc.function.name}({args})")
            out = add_task(**args)
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out, ensure_ascii=False)})
    return "(반복 한도 초과)"

print(run_agent("'서빙 강의 복습'을 할 일로 추가해줘"))
print("TASKS:", TASKS)


### 확인할 것

- `[로컬 모델 → 도구]` 로그가 찍히면 — **tool_calls 가 네이티브로 온 것**이다. 로컬 모델이 에이전트로 동작한다.
- 도구 호출이 본문 텍스트(`<tool_call>…`)로 섞여 나오면, 그 모델이 도구 호출을 지원하지 않거나 서빙이 파싱을 안 하는 것이다 — DGX에서 겪었던 그 증상이다.

### 비교 표를 채운다

| | NVIDIA API | Ollama (T4) |
|---|---|---|
| 첫 토큰까지 | | |
| 생성 속도 | | |
| 도구 호출 | | |
| rate limit | 있음 | 없음 |

> 같은 코드가 두 서빙에서 돈다 — OpenAI 호환 `/v1` 규격 덕분이다.


## Lab 3 (선택) · vLLM으로 같은 모델 서빙

강의에서 "사실상 표준"이라던 vLLM을 같은 T4에서 띄워 본다. 설치 몇 분 + 모델 다운로드(~8GB)라 시간 여유가 있을 때 한다.

- **먼저 Ollama 모델을 내린다** — vLLM이 GPU 메모리 대부분을 선점하므로 남아 있으면 충돌한다.
- T4는 BF16 미지원이라 `--dtype half`가 필수다.
- `--max-model-len` 으로 컨텍스트를 줄인다 — 가중치(약 8GB) 외 **KV 캐시가 별도**라는 사양표의 주의가 여기서 실제로 걸린다.

In [ ]:
!ollama stop qwen3:4b-instruct   # VRAM 반환

In [ ]:
%pip install -q vllm   # 수 분 소요

In [ ]:
# vLLM 서버 기동 — 다운로드+로딩까지 수 분
import subprocess
log = open("vllm.log", "w")
subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen3-4B-Instruct-2507",
     "--dtype", "half", "--max-model-len", "8192"],
    stdout=log, stderr=log)

In [ ]:
# 준비될 때까지 대기 (:8000/v1 이 열리면 완료)
import time, urllib.request
for i in range(60):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
        print("vLLM 준비 완료"); break
    except Exception:
        time.sleep(10); print(f"대기 중… {(i+1)*10}s (로그: vllm.log)")

In [ ]:
# 같은 모델, 다른 엔진 — 앱 코드는 base_url·model 문자열만 다르다
from openai import OpenAI
vllm_client = OpenAI(base_url="http://localhost:8000/v1", api_key="vllm")
r = vllm_client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=[{"role": "user", "content": "PagedAttention을 두 문장으로."}],
)
print(r.choices[0].message.content)

> Ollama(개인용)와 vLLM(서버용)을 같은 코드로 호출했다 — 비교표의 "앱 쪽 차이는 base_url 하나"가 이것이다.
>
> 이후 Lab 4로 넘어가기 전, vLLM이 VRAM을 점유 중이면 **런타임 재시작 후 Lab 1만 다시 실행**하는 게 간단하다. (Lab 4는 관리형 API를 쓰므로 GPU 없이도 된다.)

## Lab 4 · LangGraph 오케스트레이터 — 플래너·워커·종합을 코드로

9강에서 하네스(Claude Code 팀)로 했던 오케스트레이터-워커 패턴을, 오늘 배운 서빙 위에서 **LangGraph 코드로** 직접 만든다.

```
START → [planner] → Send 팬아웃 → [worker]×N (병렬) → [aggregate] → END
```

- 프롬프트가 아니라 **그래프 엣지가 순서를 강제**한다 — 어길 방법이 없다.
- `max_concurrency` 설정 하나가 9강의 "동시성 캡"을 대신한다.
- 이 랩은 NVIDIA API 키만 있으면 된다 (로컬 하이브리드는 마지막에 선택).


In [ ]:
%pip install -q langgraph openai


In [ ]:
# LLM 준비 — NVIDIA Qwen (7강과 같은 방식)
import os, getpass, operator
from typing import Annotated, TypedDict
from openai import OpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...): ")
llm = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVIDIA_API_KEY)
MODEL = "qwen/qwen3-next-80b-a3b-instruct"

def ask(prompt: str, system: str = "간결한 한국어로 답하라.") -> str:
    r = llm.chat.completions.create(
        model=MODEL, temperature=0.2, max_tokens=600,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return r.choices[0].message.content

print("LLM 준비:", MODEL)


In [ ]:
# 그래프 정의 — 상태 · 플래너 · 팬아웃 · 워커 · 종합
class State(TypedDict):
    goal: str
    tasks: list[str]
    results: Annotated[list[str], operator.add]   # 병렬 결과를 이어 붙이는 reducer
    final: str

class WorkerState(TypedDict):
    task: str

def planner(state: State):
    out = ask(f"목표를 독립적으로 처리 가능한 3개의 작은 작업으로 나눠라. 한 줄에 하나씩, 번호 없이.\n목표: {state['goal']}",
              system="계획만 세운다. 실행하지 않는다.")
    tasks = [l.strip("-· ").strip() for l in out.splitlines() if l.strip()][:3]
    print("계획:", tasks)
    return {"tasks": tasks}

def fan_out(state: State):
    # 작업 수만큼 worker 를 띄운다 — 이 리스트가 곧 병렬이다
    return [Send("worker", {"task": t}) for t in state["tasks"]]

def worker(state: WorkerState):
    print(f"  [worker 시작] {state['task'][:30]}…")
    ans = ask(f"다음 작업을 수행하라: {state['task']}")
    return {"results": [f"[{state['task']}]\n{ans}"]}

def aggregate(state: State):
    joined = "\n\n".join(state["results"])
    final = ask(f"아래 결과들을 하나의 보고로 종합하라. 중복은 정리하고 다음 단계로 마무리하라.\n\n{joined}")
    return {"final": final}

g = StateGraph(State)
g.add_node("planner", planner)
g.add_node("worker", worker)
g.add_node("aggregate", aggregate)
g.add_edge(START, "planner")
g.add_conditional_edges("planner", fan_out, ["worker"])   # 동적 팬아웃
g.add_edge("worker", "aggregate")
g.add_edge("aggregate", END)
app = g.compile()
print("그래프 컴파일 완료")


In [ ]:
# 그래프 시각화 — 조립한 구조를 눈으로 확인한다 (mermaid.ink 렌더링)
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))
# 점선(planner -.-> worker)이 Send 팬아웃 지점이다.
# 정적 그림에는 worker가 하나지만, 실행 시 작업 수만큼 늘어난다.

In [ ]:
# 실행 — max_concurrency 가 '동시성 캡'을 한 줄로 해결한다
out = app.invoke(
    {"goal": "AI 코딩 에이전트 도입 보고서 작성 — 현황 요약, 위험 요소, 다음 단계", "results": []},
    config={"max_concurrency": 2},   # 관리형 키의 호출 한도 대응 (9강 '호출 한도 아래의 설계')
)
print("\n===== 최종 보고 =====\n")
print(out["final"])


### 하이브리드 — 워커만 로컬 모델로 (선택)

Lab 1의 Ollama가 떠 있다면 (Lab 3를 했다면 런타임 재시작 후 Lab 1 재실행), **워커의 base_url만** 바꿔 병렬 호출을 관리형 키에서 로컬로 옮긴다. 그래프 코드는 그대로다.

```python
llm_local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL_LOCAL = "qwen3:4b-instruct"

def ask_local(prompt, system="간결한 한국어로 답하라."):
    r = llm_local.chat.completions.create(
        model=MODEL_LOCAL, temperature=0.2,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}])
    return r.choices[0].message.content

# worker 안의 ask → ask_local 로 바꾸고 그래프를 다시 compile 하면
# 병렬 워커의 호출량이 관리형 키에서 사라진다.
# 플래너·종합은 관리형(품질), 워커는 로컬(물량) — CH4 하이브리드 라우팅의 구현이다.
```

### 9강과의 비교 — 표를 채운다

| | 9강 하네스 (Claude Code 팀) | 10강 코드 (LangGraph) |
|---|---|---|
| 순서 강제 | 프롬프트 + agent teams | 그래프 엣지 (코드) |
| 병렬 | worktree · 서브에이전트 | Send 팬아웃 |
| 동시성 제한 | (수동) | `max_concurrency` |
| 모델 | Claude 구독 | 내가 고른 서빙 (교체 자유) |

> 같은 오케스트레이터-워커 패턴이다. 하네스는 개발 작업을, 이 그래프는 런타임 작업을 오케스트레이션한다.


## 실무 예제 · 고객 리뷰 일괄 분석

Lab 4의 그래프 구조(분할 → 팬아웃 → 병렬 → 종합)에 입력과 워커 프롬프트만 바꾼다.
분할 노드는 LLM이 아니라 일반 코드다 — 규칙으로 나눌 수 있으면 LLM을 쓰지 않는다.

실측 (Ollama `qwen3:4b-instruct` · `max_concurrency=2`): 리뷰 12건 · 3배치 병렬.


In [ ]:
# 실무 예제 — 리뷰 12건을 3배치로 병렬 분석해 경영 보고를 만든다
REVIEWS = [
    "배송이 이틀이나 늦었어요. 포장도 찌그러져 있었습니다.",
    "가격 대비 품질이 훌륭합니다. 재구매 의사 있어요.",
    "앱에서 결제가 두 번 됐는데 환불 처리가 느립니다.",
    "고객센터 상담원이 친절하게 해결해 주셨어요.",
    "제품 설명과 실제 색상이 달라요. 실망입니다.",
    "설치 기사님이 시간 약속을 정확히 지켜주셨습니다.",
    "쿠폰 적용이 안 되는 버그가 있어요. 두 번 시도했는데 실패.",
    "배송 알림이 실시간으로 와서 편했습니다.",
    "부품이 하나 빠져 있었는데 교환 절차가 복잡했습니다.",
    "디자인이 사진보다 실물이 더 좋네요.",
    "구독 해지 버튼을 찾기가 너무 어렵습니다.",
    "지난번 문의했던 불량 건, 빠르게 새 제품 보내주셨어요.",
]

class RState(TypedDict):
    reviews: list[str]
    batches: list[list[str]]
    results: Annotated[list[str], operator.add]
    final: str

class RWorker(TypedDict):
    batch: list[str]

def split(state: RState):                 # 분할은 코드로 — LLM 불필요
    rs = state["reviews"]
    return {"batches": [rs[i:i+4] for i in range(0, len(rs), 4)]}

def r_fan_out(state: RState):
    return [Send("worker", {"batch": b}) for b in state["batches"]]

def r_worker(state: RWorker):
    joined = "\n".join(f"- {r}" for r in state["batch"])
    ans = ask("다음 고객 리뷰들을 분석하라. 각 리뷰의 감성(긍정/부정)과 "
              "주제(배송·품질·결제·CS·UX 중)를 표로, 마지막에 배치 요약 한 줄.\n" + joined)
    return {"results": [ans]}

def r_aggregate(state: RState):
    joined = "\n\n".join(state["results"])
    final = ask("아래 배치별 분석을 종합해 경영 보고를 작성하라. "
                "구성: 긍/부정 비율, 주제별 핵심 불만 2가지, 개선 우선순위 2가지.\n\n" + joined)
    return {"final": final}

rg = StateGraph(RState)
rg.add_node("split", split)
rg.add_node("worker", r_worker)
rg.add_node("aggregate", r_aggregate)
rg.add_edge(START, "split")
rg.add_conditional_edges("split", r_fan_out, ["worker"])
rg.add_edge("worker", "aggregate")
rg.add_edge("aggregate", END)
rapp = rg.compile()

out = rapp.invoke({"reviews": REVIEWS, "results": []}, config={"max_concurrency": 2})
print(out["final"])


## 체크리스트

**Lab 0~1 — 서빙**
- [ ] T4 GPU 런타임을 확인했다
- [ ] Colab에 Ollama를 띄우고 `qwen3:4b-instruct`를 받았다
- [ ] `localhost:11434/v1` 을 openai 패키지로 호출했다 — NVIDIA API와 같은 규격
- [ ] 스트리밍이 동작하고, `nvidia-smi`에서 VRAM 사용을 확인했다

**Lab 2 — 에이전트 연결**
- [ ] 7강 루프를 base_url 두 줄 교체로 내 서버에 붙였다
- [ ] tool_calls 가 네이티브로 오는 것을 확인했다
- [ ] NVIDIA API와 속도·품질을 비교해 표를 채웠다

**Lab 3 (선택) — 엔진 교체**
- [ ] vLLM으로 같은 모델을 서빙하고 base_url만 바꿔 호출했다
- [ ] `--dtype half` · `--max-model-len` 이 왜 필요한지 안다 (T4 제약 · KV 캐시 예산)

**Lab 4 — LangGraph 오케스트레이터**
- [ ] planner → Send 팬아웃 → worker 병렬 → aggregate 그래프가 돌았다
- [ ] 그래프 시각화에서 점선(팬아웃 지점)을 확인했다
- [ ] `max_concurrency=2` 로 동시성 제한을 확인했다
- [ ] (선택) 워커를 Ollama로 바꿔 하이브리드 라우팅을 확인했다

**개념**
- [ ] KV 캐시 · PagedAttention · continuous batching · 양자화를 각각 한 문장으로 설명할 수 있다
- [ ] Ollama / vLLM / NIM 이 각각 어떤 상황의 선택인지 안다

> 관리형 API의 제약(호출 한도·데이터 위치)은 자가 서빙으로 대체할 수 있는 선택 사항이다.

## 부록 · 내 컴퓨터에 설치 (수업 밖 자습)

Colab 세션은 초기화되므로, 계속 쓰려면 내 컴퓨터에 설치한다. GPU가 없어도 `qwen3:4b-instruct`(Q4, 약 2.5GB)는 RAM 16GB에서 동작한다.

```powershell
# Windows
winget install Ollama.Ollama
```
```bash
# macOS
brew install ollama
# Linux
curl -fsSL https://ollama.com/install.sh | sh
```

설치 후 새 터미널에서:

```bash
ollama pull qwen3:4b-instruct   # 저사양(RAM 8GB)은 qwen2.5:3b
ollama run qwen3:4b-instruct    # 터미널 대화 (종료: /bye)
```

이후 이 노트북의 모든 코드가 로컬 Jupyter·VS Code에서 그대로 돈다 — `localhost:11434/v1` 이 같기 때문이다.

9강 앱(mcp-host)에 로컬 모델을 붙이려면 모델 스위처에 항목을 더한다:

```ts
// lib/models.ts
{ id: "qwen3:4b-instruct",
  baseURL: "http://localhost:11434/v1",
  label: "로컬 Ollama" }
```